# 3. Cell annotation and Gaussian robustness

Run from the repository root with the **MetacellFormer-env** Python 3.11 kernel **after notebook 02**. The downloaded PBMC_368K input, roster, and model files from notebook 02 are sufficient; no additional downloads are needed. The encoder receives individual single cells. This notebook scores their embeddings and demonstrates one Gaussian condition (ratio 0.1, draw seed 42) with a classifier fitted on clean training cells.

**CPU** works (about 3 minutes on the tested two-thread CPU); change `device` to `cuda:0` for GPU embedding extraction. Expected files under `example_outputs/PBMC_368K/`: `cell_annotation/per_dataset_scores.tsv` (one score row), `noise/{train_clean,clean,gaussian_0.1_s42}.h5ad`, three corresponding `robustness_embeddings/*.npz` files (48 × 1,024 each), and `robustness_scores/all_scores.tsv` (three probe-seed rows). Completion means all 48 test cells are covered, none fail, and all three rows contain finite scores.

On this fixed **96-cell example**, annotation/clean macro-F1 is about 0.876 and noisy macro-F1 about 0.610. These are example scores; the paper's results use the full evaluation roster.

In [1]:
from pathlib import Path
import os
import sys
import subprocess
import pandas as pd

repo = Path.cwd().resolve()
assert (repo / "benchmarks/tasks/cell_annotation.py").is_file(), "Start Jupyter from the repository root"
data_root = Path(os.environ.get("METACELLFORMER_EVALUATION", "../MetacellFormer-evaluation")).expanduser().resolve()
work = repo / "example_outputs/PBMC_368K"
roster_file = work / "example_roster.json"
source = data_root / "inputs/cell_annotation/PBMC_368K.h5ad"
config = work / "runtime.json"
assert all(p.is_file() for p in (roster_file, source, config, work / "train.npz", work / "test.npz")), "Run notebook 02 first"
device = "cpu"  # Match notebook 02, or change to cuda:0.

In [2]:
subprocess.run([sys.executable, "-m", "benchmarks.tasks.cell_annotation",
                "--train-embedding", str(work / "train.npz"),
                "--test-embedding", str(work / "test.npz"),
                "--roster", str(roster_file), "--model", "MetacellFormer",
                "--dataset", "PBMC_368K", "--output-dir", str(work / "cell_annotation")],
               cwd=repo, check=True)
print(pd.read_csv(work / "cell_annotation/per_dataset_scores.tsv", sep="\t"))

         model_id    dataset  metric_primary
0  MetacellFormer  PBMC_368K        0.875699


In [3]:
noise = work / "noise"
subprocess.run([sys.executable, "-m", "benchmarks.tasks.robustness", "prepare",
                "--input", str(source), "--roster", str(roster_file),
                "--dataset", "PBMC_368K", "--ratios", "0.1", "--seeds", "42",
                "--output-dir", str(noise)], cwd=repo, check=True)
robust_embeddings = work / "robustness_embeddings"
robust_embeddings.mkdir(parents=True, exist_ok=True)
for name in ("train_clean", "clean", "gaussian_0.1_s42"):
    subprocess.run([sys.executable, "-m", "benchmarks.embeddings.extract_embeddings",
                    "--input", str(noise / f"{name}.h5ad"),
                    "--output", str(robust_embeddings / f"{name}.npz"),
                    "--config", str(config), "--model", "MetacellFormer",
                    "--task", "robustness", "--device", device, "--batch-size", "4"],
                   cwd=repo, check=True)
subprocess.run([sys.executable, "-m", "benchmarks.tasks.robustness", "score",
                "--roster", str(roster_file), "--embedding-dir", str(robust_embeddings),
                "--model", "MetacellFormer", "--dataset", "PBMC_368K",
                "--ratios", "0.1", "--seeds", "42",
                "--output-dir", str(work / "robustness_scores")], cwd=repo, check=True)
print(pd.read_csv(work / "robustness_scores/all_scores.tsv", sep="\t")[["probe_seed", "macro_f1", "clean_macro_f1", "retention"]])

   probe_seed  macro_f1  clean_macro_f1  retention
0           1  0.609513        0.875699    0.69603
1           2  0.609513        0.875699    0.69603
2           3  0.609513        0.875699    0.69603
